In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# For embeddings
from sentence_transformers import SentenceTransformer

# For vector database
import chromadb

# For LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

print("All libraries imported successfully!")

All libraries imported successfully!


In [10]:
print("=" * 50)
print("LOADING VECTOR STORE")
print("=" * 50)

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="../vector_store/chroma_db")

# Get the collection
collection_name = "complaints"
collection = chroma_client.get_collection(name=collection_name)

print(f"Collection loaded: {collection_name}")
print(f"Total vectors: {collection.count()}")

# Load the embedding model (SENTENCE TRANSFORMER)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # ← RENAMED to embedding_model
print(f" Embedding model loaded (384-dim)")

LOADING VECTOR STORE
Collection loaded: complaints
Total vectors: 28866


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Embedding model loaded (384-dim)


In [11]:
print("=" * 50)
print("BUILDING RETRIEVER")
print("=" * 50)

def retrieve_chunks(query, n_results=5):
    """
    Retrieve relevant chunks from the vector store
    
    Args:
        query (str): User's question
        n_results (int): Number of chunks to retrieve
    
    Returns:
        dict: Retrieved chunks with metadata
    """
    # Generate query embedding using embedding_model (NOT phi-2)
    query_embedding = embedding_model.encode(query).tolist()  # ← Changed to embedding_model
    
    # Search the vector store
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    
    # Format results
    retrieved = {
        'documents': results['documents'][0],
        'metadatas': results['metadatas'][0],
        'distances': results['distances'][0]
    }
    
    return retrieved

# Test the retriever
test_query = "Why are customers unhappy with credit cards?"
results = retrieve_chunks(test_query, n_results=3)

print(f"Query: {test_query}")
print("\nTop 3 results:")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'], 
    results['metadatas'], 
    results['distances']
)):
    print(f"\nResult {i+1} (Distance: {dist:.4f}):")
    print(f"  Product: {meta.get('product', 'Unknown')}")
    print(f"  Preview: {doc[:150]}...")

BUILDING RETRIEVER
Query: Why are customers unhappy with credit cards?

Top 3 results:

Result 1 (Distance: 0.3519):
  Product: Credit card
  Preview: first time it s like a scheme . as a good credit card company as i ve went through these so many years they ve done a lot and trying to do more and mo...

Result 2 (Distance: 0.3642):
  Product: Credit card
  Preview: t either the credit card company is not reporting correctly or the reporting companies are not reporting correctly. either way ... me the consumer is ...

Result 3 (Distance: 0.3719):
  Product: Credit card
  Preview: ists. the credit card companies and the banks are stealing money from people and then telling them after the fact and i m sure in some instances peopl...


In [4]:
def format_context(retrieved_results, max_chars=2000):
    """
    Format retrieved chunks into a context string for the LLM
    
    Args:
        retrieved_results (dict): Results from retrieve_chunks
        max_chars (int): Maximum context length
    
    Returns:
        str: Formatted context
    """
    context_parts = []
    total_chars = 0
    
    for i, (doc, meta) in enumerate(zip(
        retrieved_results['documents'], 
        retrieved_results['metadatas']
    )):
        # Create a prefix with metadata
        prefix = f"[Source {i+1}] Product: {meta.get('product', 'Unknown')}"
        if meta.get('issue'):
            prefix += f", Issue: {meta.get('issue')}"
        prefix += "\n"
        
        # Add the chunk
        chunk_text = prefix + doc + "\n"
        
        # Check if adding this exceeds max_chars
        if total_chars + len(chunk_text) > max_chars:
            break
        
        context_parts.append(chunk_text)
        total_chars += len(chunk_text)
    
    return "\n".join(context_parts)

# Test the formatter
context = format_context(results)
print(f"Context length: {len(context)} characters")
print("\nContext preview:")
print(context[:300] + "...")

Context length: 1399 characters

Context preview:
[Source 1] Product: Credit card, Issue: Closing your account
first time it s like a scheme . as a good credit card company as i ve went through these so many years they ve done a lot and trying to do more and more for their consumers to win business and win reputation it s so funny that i see a cred...


In [5]:
def create_prompt(question, context):
    """
    Create a prompt for the LLM
    
    Args:
        question (str): User's question
        context (str): Retrieved context
    
    Returns:
        str: Complete prompt
    """
    prompt = f"""You are a financial analyst assistant for CrediTrust Financial. Your task is to answer questions about customer complaints.

Instructions:
1. Use ONLY the provided context to answer the question
2. If the context doesn't contain enough information, say "I don't have enough information to answer this question"
3. Be concise and specific
4. Cite the source product when relevant
5. Do not make up information

Context:
{context}

Question: {question}

Answer:"""
    
    return prompt

# Test the prompt
sample_question = "Why are customers unhappy with credit cards?"
sample_prompt = create_prompt(sample_question, context)
print("Prompt preview:")
print(sample_prompt[:400] + "...")

Prompt preview:
You are a financial analyst assistant for CrediTrust Financial. Your task is to answer questions about customer complaints.

Instructions:
1. Use ONLY the provided context to answer the question
2. If the context doesn't contain enough information, say "I don't have enough information to answer this question"
3. Be concise and specific
4. Cite the source product when relevant
5. Do not make up inf...


In [12]:
print("=" * 50)
print("SETTING UP LLM - Phi-2")
print("=" * 50)

model_name = "microsoft/phi-2"

print(f"Loading model: {model_name}")

# Load tokenizer and model (THIS IS THE LLM, NOT THE EMBEDDING MODEL)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(  # ← RENAMED to llm_model
    model_name,
    trust_remote_code=True,
    torch_dtype="auto"
)

# Create a pipeline using llm_model
generator = pipeline(
    "text-generation",
    model=llm_model,  # ← Changed to llm_model
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(f"✅ Model loaded: {model_name}")

SETTING UP LLM - Phi-2
Loading model: microsoft/phi-2


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

✅ Model loaded: microsoft/phi-2


In [13]:
# Quick verification cell
print("=" * 50)
print("VERIFYING SETUP")
print("=" * 50)

# Check if model is loaded
try:
    print(f"✅ Model: {model_name}")
    print(f"✅ Tokenizer: loaded")
    print(f"✅ Generator: ready")
except:
    print("❌ Something isn't loaded properly")
    print("Please run Cell 6 again")

VERIFYING SETUP
✅ Model: microsoft/phi-2
✅ Tokenizer: loaded
✅ Generator: ready


In [14]:
print("=" * 50)
print("BUILDING RAG PIPELINE")
print("=" * 50)

def rag_pipeline(question, n_results=5, max_context_chars=2000):
    """
    Complete RAG pipeline
    
    Args:
        question (str): User's question
        n_results (int): Number of chunks to retrieve
        max_context_chars (int): Maximum context length
    
    Returns:
        dict: {
            'question': str,
            'answer': str,
            'sources': list,
            'context': str
        }
    """
    # Step 1: Retrieve relevant chunks (uses embedding_model)
    retrieved = retrieve_chunks(question, n_results=n_results)
    
    # Step 2: Format context
    context = format_context(retrieved, max_chars=max_context_chars)
    
    # Step 3: Create prompt
    prompt = create_prompt(question, context)
    
    # Step 4: Generate answer (uses generator from Cell 6)
    response = generator(prompt, max_new_tokens=300)[0]['generated_text']
    
    # Extract just the answer part
    answer = response.split("Answer:")[-1].strip()
    
    return {
        'question': question,
        'answer': answer,
        'sources': retrieved['documents'],
        'source_metadata': retrieved['metadatas'],
        'context': context,
        'full_prompt': prompt
    }

# Test the pipeline
test_question = "Why are customers unhappy with credit cards?"
print(f"Testing with: {test_question}")
print("-" * 50)

result = rag_pipeline(test_question)

print(f"\n📝 Question: {result['question']}")
print(f"\n💡 Answer: {result['answer']}")
print(f"\n📚 Sources: {len(result['sources'])} chunks retrieved")

# Show first source preview
if result['sources']:
    print(f"\n📄 First source preview:")
    print(f"   Product: {result['source_metadata'][0].get('product', 'Unknown')}")
    print(f"   Preview: {result['sources'][0][:150]}...")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BUILDING RAG PIPELINE
Testing with: Why are customers unhappy with credit cards?
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



📝 Question: Why are customers unhappy with credit cards?

💡 Answer: Customers are unhappy with credit cards because of various issues including the closing of accounts, incorrect reporting, unfair practices, and deceptive advertising and marketing. They feel that the companies are not transparent and are taking advantage of their customers' financial situations.

📚 Sources: 5 chunks retrieved

📄 First source preview:
   Product: Credit card
   Preview: first time it s like a scheme . as a good credit card company as i ve went through these so many years they ve done a lot and trying to do more and mo...


In [15]:
print("=" * 50)
print("TESTING RAG PIPELINE WITH MULTIPLE QUESTIONS")
print("=" * 50)

# Define test questions covering different products
test_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the main issues with savings accounts?",
    "Why do customers complain about money transfers?",
    "What problems do people have with personal loans?",
    "Are there any fraud-related complaints?"
]

# Store results
all_results = []

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"Test {i}: {q}")
    print('-' * 50)
    
    try:
        result = rag_pipeline(q)
        all_results.append(result)
        
        print(f"\n📝 Question: {result['question']}")
        print(f"\n💡 Answer: {result['answer']}")
        print(f"\n📚 Sources: {len(result['sources'])} chunks retrieved")
        
        # Show product distribution of sources
        products = [m.get('product', 'Unknown') for m in result['source_metadata']]
        print(f"📊 Products in sources: {', '.join(set(products))}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        all_results.append({'question': q, 'answer': f"Error: {e}", 'sources': []})

print("\n" + "=" * 50)
print("✅ All tests completed!")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTING RAG PIPELINE WITH MULTIPLE QUESTIONS

Test 1: Why are customers unhappy with credit cards?
--------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Question: Why are customers unhappy with credit cards?

💡 Answer: Customers are unhappy with credit cards because of various issues such as closing their account, inaccurate reporting by credit card companies or reporting companies, unfair advertising and marketing practices, and refusal to work with customers in financial hardship.

📚 Sources: 5 chunks retrieved
📊 Products in sources: Credit card

Test 2: What are the main issues with savings accounts?
--------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Question: What are the main issues with savings accounts?

💡 Answer: The main issues with savings accounts are the delay in transferring funds, denying access to credit and savings funds, rude and unprofessional customer service, unjust account freezes, and the resulting financial hardships due to late fees, disconnection fees, and inability to purchase essential necessities.

📚 Sources: 5 chunks retrieved
📊 Products in sources: Savings account, Personal loan

Test 3: Why do customers complain about money transfers?
--------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Question: Why do customers complain about money transfers?

💡 Answer: Customers complain about money transfers because the money is not available when promised. According to Source 1, the company has the ability and technology to instantly cancel a transaction, but they have not done so, which is dishonest and disingenuous.

📚 Sources: 5 chunks retrieved
📊 Products in sources: Savings account, Credit card, Money transfer

Test 4: What problems do people have with personal loans?
--------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Question: What problems do people have with personal loans?

💡 Answer: People have problems with personal loans because they struggle to pay them back, they are charged unexpected fees or interest, and they are not treated fairly by the company.

📚 Sources: 5 chunks retrieved
📊 Products in sources: Savings account, Credit card, Personal loan

Test 5: Are there any fraud-related complaints?
--------------------------------------------------

📝 Question: Are there any fraud-related complaints?

💡 Answer: The issue with the credit card is not fraud-related, as mentioned in [Source 4]. The customer has a problem with a purchase shown on their statement, but no mention of fraud is made.

📚 Sources: 5 chunks retrieved
📊 Products in sources: Savings account, Credit card, Money transfer

✅ All tests completed!


In [16]:
print("=" * 50)
print("EVALUATION TABLE")
print("=" * 50)

import pandas as pd

# Define evaluation questions (10 questions)
evaluation_questions = [
    {"question": "What are the most common complaints about credit cards?", "category": "Credit Cards"},
    {"question": "Why do customers complain about money transfers?", "category": "Money Transfers"},
    {"question": "What issues do customers face with savings accounts?", "category": "Savings Accounts"},
    {"question": "What problems do people have with personal loans?", "category": "Personal Loans"},
    {"question": "Are there any fraud-related complaints?", "category": "Cross-Product"},
    {"question": "How do complaints vary by state?", "category": "Cross-Product"},
    {"question": "What are the top issues across all products?", "category": "Cross-Product"},
    {"question": "Are there complaints about specific companies?", "category": "Cross-Product"},
    {"question": "What billing issues do customers report?", "category": "Cross-Product"},
    {"question": "How has complaint volume changed over time?", "category": "Cross-Product"}
]

# Run evaluation
evaluation_results = []

for item in evaluation_questions:
    print(f"Processing: {item['question'][:50]}...")
    
    try:
        result = rag_pipeline(item['question'])
        
        # Quality scoring
        quality_score = 3  # Default
        if len(result['answer']) > 100 and "don't have enough" not in result['answer'].lower():
            quality_score = 5
        elif len(result['answer']) > 50:
            quality_score = 4
        elif "don't have enough" in result['answer'].lower():
            quality_score = 2
        
        # Get products from sources
        products = list(set([m.get('product', 'Unknown') for m in result['source_metadata']]))
        
        evaluation_results.append({
            'Question': item['question'][:60] + "..." if len(item['question']) > 60 else item['question'],
            'Category': item['category'],
            'Answer Preview': result['answer'][:150] + "..." if len(result['answer']) > 150 else result['answer'],
            'Sources Used': len(result['sources']),
            'Products in Sources': ', '.join(products[:3]),
            'Quality Score (1-5)': quality_score,
            'Comments': 'Good' if quality_score >= 4 else 'Needs improvement'
        })
        
    except Exception as e:
        evaluation_results.append({
            'Question': item['question'][:60] + "..." if len(item['question']) > 60 else item['question'],
            'Category': item['category'],
            'Answer Preview': f"Error: {str(e)[:50]}",
            'Sources Used': 0,
            'Products in Sources': 'N/A',
            'Quality Score (1-5)': 1,
            'Comments': 'Failed to generate'
        })

# Create DataFrame
eval_df = pd.DataFrame(evaluation_results)

print("\n" + "=" * 50)
print("EVALUATION TABLE")
print("=" * 50)
print(eval_df.to_string(index=False))

# Save to CSV
eval_df.to_csv('../data/processed/evaluation_results.csv', index=False)
print("\n✅ Evaluation results saved to: data/processed/evaluation_results.csv")

# Summary statistics
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"Total questions evaluated: {len(eval_df)}")
print(f"Average quality score: {eval_df['Quality Score (1-5)'].mean():.2f}/5")
print(f"Questions scoring 4+: {len(eval_df[eval_df['Quality Score (1-5)'] >= 4])}")
print(f"Questions with errors: {len(eval_df[eval_df['Quality Score (1-5)'] == 1])}")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EVALUATION TABLE
Processing: What are the most common complaints about credit c...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: Why do customers complain about money transfers?...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Processing: Are there complaints about specific companies?...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: What billing issues do customers report?...


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: How has complaint volume changed over time?...

EVALUATION TABLE
                                               Question         Category                                                                                                                                             Answer Preview  Sources Used                          Products in Sources  Quality Score (1-5) Comments
What are the most common complaints about credit cards?     Credit Cards  The most common complaints about credit cards are regarding the payment process, incorrect reporting of payments, negative impact on credit scores, mi...             5                                  Credit card                    5     Good
       Why do customers complain about money transfers?  Money Transfers  Customers complain about money transfers because they have not received the promised amount of money. They believe that the company has the ability an...             5 Savings account, Credit card, Money transfer   

In [17]:
print("=" * 50)
print("TASK 3 COMPLETE - SUMMARY")
print("=" * 50)

print("\n✅ What we built:")
print("  - Retriever: Semantic search over 28,866 complaint chunks")
print("  - Prompt Engine: Structured prompts for consistent answers")
print("  - LLM: Phi-2 (2.7B parameters) for answer generation")
print("  - RAG Pipeline: End-to-end question answering")

print("\n📊 Test Results:")
print(f"  - Questions tested: {len(evaluation_results)}")
print(f"  - Average quality: {eval_df['Quality Score (1-5)'].mean():.2f}/5")

print("\n📁 Files Saved:")
print("  - data/processed/evaluation_results.csv")
print("  - data/processed/rag_sample_outputs.csv")

print("\n✅ Task 3 completed successfully!")

# Show sample question and answer
print("\n" + "=" * 50)
print("SAMPLE OUTPUT")
print("=" * 50)
if all_results:
    sample = all_results[0]
    print(f"Q: {sample['question']}")
    print(f"A: {sample['answer']}")

TASK 3 COMPLETE - SUMMARY

✅ What we built:
  - Retriever: Semantic search over 28,866 complaint chunks
  - Prompt Engine: Structured prompts for consistent answers
  - LLM: Phi-2 (2.7B parameters) for answer generation
  - RAG Pipeline: End-to-end question answering

📊 Test Results:
  - Questions tested: 10
  - Average quality: 4.70/5

📁 Files Saved:
  - data/processed/evaluation_results.csv
  - data/processed/rag_sample_outputs.csv

✅ Task 3 completed successfully!

SAMPLE OUTPUT
Q: Why are customers unhappy with credit cards?
A: Customers are unhappy with credit cards because of various issues such as closing their account, inaccurate reporting by credit card companies or reporting companies, unfair advertising and marketing practices, and refusal to work with customers in financial hardship.
